# Chapter 27: What We Did Not Touch

**Source orientation.** Sections 27.1-27.4; printed pages 525-556; PDF pages 547-578. The source span closes the book by pointing beyond the earlier tour: algebraic projective geometry, oriented/discrete projective geometry, quantum information, and dynamic projective geometry. This notebook is original teaching prose and synthetic computation; it does not use textbook figures, page crops, or copied exercises.

**Chapter goal.** Build a working computational map for the topics the book explicitly leaves open: count intersections of higher-degree curves, see Hessians and tangent duality, encode incidence as signs, treat quantum states as points of complex projective spaces, and model dynamic constructions by tracing complex branches.

The chapter is not a grab bag. Each section below takes a familiar projective habit from earlier chapters and shows what it becomes at the frontier: determinants become chirotopes, conics become algebraic curves and dual curves, CP1 becomes a qubit state space, and ambiguous construction primitives become analytic branches that must be followed through motion history.


In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
    nested = candidate / "Perspectives-on-Projective-Geometry"
    if (nested / "AGENTS.md").exists() and (nested / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = nested
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the Perspectives on Projective Geometry course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-27-what-we-did-not-touch"
for child in ("figures", "html", "checks", "tables"):
    (ARTIFACT_ROOT / child).mkdir(parents=True, exist_ok=True)

BOOK_ROOT, ARTIFACT_ROOT


In [ ]:
import json
from itertools import combinations

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from IPython.display import display
from plotly.subplots import make_subplots

from utils.artifacts import assert_artifacts, book_relative, display_artifact, image_stats, save_json, save_table

plt.ioff()
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
CHECKS = ARTIFACT_ROOT / "checks"
TABLES = ARTIFACT_ROOT / "tables"


def save_figure(fig, filename):
    path = FIGURES / filename
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def load_check(filename):
    return json.loads((CHECKS / filename).read_text(encoding="utf-8"))


def sign_symbol(value, tol=1e-9):
    value = float(np.real_if_close(value))
    if value > tol:
        return "+"
    if value < -tol:
        return "-"
    return "0"


storyboard = json.loads((CHECKS / "storyboard.json").read_text(encoding="utf-8"))
pd.DataFrame(storyboard["visual sequence"])[["order", "concept", "artifact filename", "validation/invariant"]]


## Computational Translation Guide

| Source topic | Computational object | Inspection target |
| --- | --- | --- |
| Algebraic projective curves | homogeneous polynomial equations in `[x:y:z]` | degree, parameter count, singularities, and intersections counted over complex projective space |
| Bezout and Cayley-Bacharach | polynomial roots with multiplicity; common zeros of curves | why a generic line meets a cubic three times and a tangent contact counts twice |
| Hessians and duality | determinant of second derivatives; tangent-line coordinates | inflection points as common zeros of a curve and its Hessian; tangents as points in the dual plane |
| Oriented matroids | signs of determinants and signs of point-line separation | incidences appear as zeros, relative positions as plus/minus signs, and realizability as a hard question |
| Quantum states | complex homogeneous coordinates modulo nonzero scalar | a qubit is CP1, two qubits live in CP3, and entanglement is an invariant of restricted local projective actions |
| Dynamic geometry | construction steps as relations with analytic branches | branch choices depend on motion history; complex detours avoid critical values |

The key shift is that diagrams become carriers of algebraic state. A picture may suggest a relation, but the check records whether the relevant invariant actually survived the projective translation.


## Algebraic Curves And Bezout

Chapter 27 starts by leaving the world of lines and conics. A plane curve of degree `d` is a homogeneous degree-`d` polynomial equation. Bezout's theorem says that two curves of degrees `m` and `n` meet in `m*n` projective points once multiplicity, complex points, and points at infinity are counted, unless the curves share a component.

The smallest check worth doing is a line against a cubic. A generic line gives a cubic equation and therefore three complex roots. A tangent line still gives three roots, but one root occurs twice. That double root is the computational version of the source's warning that tangencies must be counted with multiplicity.


In [ ]:
# Projective cubic: Y^2 Z - X^3 + X Z^2 = 0.
x = sp.symbols("x")
m = sp.Rational(2, 5)
b = sp.Rational(1, 5)
generic_poly = sp.expand((m * x + b) ** 2 - x**3 + x)
generic_roots = [complex(r) for r in sp.nroots(generic_poly)]

x0 = sp.Rational(3, 2)
y0 = sp.sqrt(x0**3 - x0)
tangent_m = sp.simplify((3 * x0**2 - 1) / (2 * y0))
tangent_b = sp.simplify(y0 - tangent_m * x0)
tangent_poly = sp.expand((tangent_m * x + tangent_b) ** 2 - x**3 + x)
tangent_factor = sp.factor(tangent_poly)
base_root_multiplicity = 0
quotient = sp.Poly(tangent_poly, x)
while quotient.degree() >= 0:
    q, r = sp.div(quotient, sp.Poly(x - x0, x))
    if r.as_expr() == 0:
        base_root_multiplicity += 1
        quotient = q
    else:
        break

xx_left = np.linspace(-1.0, 0.0, 300)
yy_left = np.sqrt(np.maximum(xx_left**3 - xx_left, 0))
xx_right = np.linspace(1.0, 2.05, 300)
yy_right = np.sqrt(np.maximum(xx_right**3 - xx_right, 0))
line_x = np.linspace(-1.1, 2.05, 300)
generic_y = float(m) * line_x + float(b)
tangent_y = float(tangent_m.evalf()) * line_x + float(tangent_b.evalf())

fig, ax = plt.subplots(figsize=(8.4, 5.4))
for xs, ys, label in [
    (xx_left, yy_left, "cubic $Y^2Z=X^3-XZ^2$"),
    (xx_right, yy_right, None),
]:
    ax.plot(xs, ys, color="#1b6ca8", lw=2.2, label=label)
    ax.plot(xs, -ys, color="#1b6ca8", lw=2.2)
ax.plot(line_x, generic_y, color="#d95f02", lw=1.8, label="generic line")
ax.plot(line_x, tangent_y, color="#7570b3", lw=1.8, label="tangent line")

real_generic_points = []
for root in generic_roots:
    if abs(root.imag) < 1e-8 and -1.1 <= root.real <= 2.05:
        real_generic_points.append((root.real, float(m) * root.real + float(b)))
if real_generic_points:
    ax.scatter(*zip(*real_generic_points), s=60, color="#d95f02", edgecolor="white", zorder=5)

base_point = (float(x0), float(y0.evalf()))
ax.scatter([base_point[0]], [base_point[1]], s=85, color="#7570b3", edgecolor="white", zorder=6)
ax.annotate(
    "double root at tangency",
    xy=base_point,
    xytext=(0.75, 1.9),
    arrowprops={"arrowstyle": "->", "color": "#444444"},
    color="#333333",
)
ax.axhline(0, color="#888888", lw=0.8)
ax.axvline(0, color="#888888", lw=0.8)
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.15, 2.1)
ax.set_ylim(-2.2, 2.25)
ax.set_xlabel("affine x = X/Z")
ax.set_ylabel("affine y = Y/Z")
ax.set_title("Bezout in one affine chart: a line meets a cubic three times")
ax.legend(loc="lower right")
root_summary = "generic roots: " + ", ".join(f"{r.real:.3f}{r.imag:+.3f}i" for r in generic_roots)
ax.text(
    0.02,
    0.02,
    root_summary + f"\ntangent multiplicity at x=3/2: {base_root_multiplicity}",
    transform=ax.transAxes,
    fontsize=8.5,
    va="bottom",
    bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "#cccccc"},
)
bezout_path = save_figure(fig, "bezout-line-cubic-intersections.png")

bezout_checks = {
    "projective_cubic": "Y^2*Z - X^3 + X*Z^2",
    "line_substitution_degree": int(sp.Poly(generic_poly, x).degree()),
    "generic_root_count_over_complex": len(generic_roots),
    "generic_roots": [[float(r.real), float(r.imag)] for r in generic_roots],
    "tangent_base_point": [float(x0), float(y0.evalf())],
    "tangent_line_slope": float(tangent_m.evalf()),
    "tangent_root_multiplicity_at_base": base_root_multiplicity,
    "tangent_factor": str(tangent_factor),
}
assert bezout_checks["line_substitution_degree"] == 3
assert bezout_checks["generic_root_count_over_complex"] == 3
assert bezout_checks["tangent_root_multiplicity_at_base"] >= 2
save_json(bezout_checks, ARTIFACT_ROOT, "checks", "bezout-line-cubic-checks.json")
book_relative(bezout_path), bezout_checks


In [ ]:
display_artifact(bezout_path, width=760)
load_check("bezout-line-cubic-checks.json")


## Hessians, Inflection Points, And Tangent Duality

The source uses cubics to show how quickly projective geometry becomes richer after conics. The Hessian of a curve is the determinant of the matrix of second partial derivatives. For a degree-`d` curve its degree is `3d - 6`, and the common zeros of the curve and its Hessian are inflection points. For a smooth cubic, that count is nine in the complex projective plane.

The same section also points to duality. A smooth point on a homogeneous curve has a tangent line whose coefficients are given by the gradient of the polynomial at that point. As the point moves along the curve, those tangent-line coefficients trace a dual curve. We sample that tangent map directly rather than trying to solve the full degree-six dual curve.


In [ ]:
X, Y, Z = sp.symbols("X Y Z")
fermat = X**3 + Y**3 + Z**3
hessian_det = sp.factor(sp.det(sp.hessian(fermat, (X, Y, Z))))
expected_hessian = 216 * X * Y * Z
assert sp.simplify(hessian_det - expected_hessian) == 0

minus_one_roots = [
    sp.Integer(-1),
    sp.Rational(1, 2) + sp.sqrt(3) * sp.I / 2,
    sp.Rational(1, 2) - sp.sqrt(3) * sp.I / 2,
]
inflection_points = []
for root in minus_one_roots:
    inflection_points.extend([(0, root, 1), (root, 0, 1), (root, 1, 0)])
inflection_vanishes = [
    sp.simplify(fermat.subs({X: a, Y: b0, Z: c})) == 0
    and sp.simplify(hessian_det.subs({X: a, Y: b0, Z: c})) == 0
    for a, b0, c in inflection_points
]

xs = np.linspace(-2.15, 1.25, 500)
ys = np.cbrt(-(xs**3 + 1.0))
sample_xs = np.array([-1.85, -1.25, -0.55, 0.15, 0.85])
sample_ys = np.cbrt(-(sample_xs**3 + 1.0))

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12.0, 5.4), gridspec_kw={"width_ratios": [1.25, 1.0]})
ax.plot(xs, ys, color="#1b6ca8", lw=2.3, label="F = X^3+Y^3+Z^3")
ax.axvline(0, color="#d95f02", lw=1.6, label="Hessian slice xy=0")
ax.axhline(0, color="#d95f02", lw=1.6)
line_x = np.linspace(-2.2, 1.25, 300)
tangent_residuals = []
for sx, sy in zip(sample_xs, sample_ys):
    grad = np.array([3 * sx**2, 3 * sy**2, 3.0])
    point = np.array([sx, sy, 1.0])
    tangent_residuals.append(abs(float(np.dot(grad, point))))
    if abs(grad[1]) > 1e-9:
        line_y = -(grad[0] * line_x + grad[2]) / grad[1]
        ax.plot(line_x, line_y, color="#666666", lw=0.9, alpha=0.65)
ax.scatter(sample_xs, sample_ys, s=38, color="#333333", zorder=5, label="sample tangent points")
ax.scatter([-1, 0], [0, -1], s=90, color="#e7298a", edgecolor="white", zorder=6, label="finite real inflections")
ax.set_xlim(-2.15, 1.2)
ax.set_ylim(-2.1, 1.9)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("x = X/Z")
ax.set_ylabel("y = Y/Z")
ax.set_title("Fermat cubic, Hessian, and tangent-line samples")
ax.legend(loc="upper right", fontsize=8)
ax.text(
    0.02,
    0.02,
    "H(F)=216XYZ; z=0 contributes three more projective points",
    transform=ax.transAxes,
    fontsize=8.3,
    va="bottom",
    bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "#cccccc"},
)

points9 = [(i, j) for i in range(3) for j in range(3)]
line_families = []
for j in range(3):
    line_families.append(("row", [(i, j) for i in range(3)]))
for i in range(3):
    line_families.append(("column", [(i, j) for j in range(3)]))
for b0 in range(3):
    line_families.append(("slope +1", [(i, (i + b0) % 3) for i in range(3)]))
for b0 in range(3):
    line_families.append(("slope -1", [(i, (-i + b0) % 3) for i in range(3)]))
colors = {"row": "#1b9e77", "column": "#d95f02", "slope +1": "#7570b3", "slope -1": "#e7298a"}
for family, pts in line_families:
    arr = np.array(pts, dtype=float)
    ax2.plot(arr[:, 0], arr[:, 1], color=colors[family], lw=1.6, alpha=0.65)
for i, j in points9:
    ax2.scatter([i], [j], s=85, color="#ffffff", edgecolor="#222222", zorder=4)
    ax2.text(i, j, f"{i}{j}", ha="center", va="center", fontsize=8, zorder=5)
ax2.set_xlim(-0.45, 2.45)
ax2.set_ylim(-0.45, 2.45)
ax2.set_xticks([0, 1, 2])
ax2.set_yticks([0, 1, 2])
ax2.set_aspect("equal")
ax2.set_title("Hesse incidence pattern: 9 points, 12 triples")
for family, color in colors.items():
    ax2.plot([], [], color=color, lw=2, label=family)
ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, fontsize=8)

hessian_path = save_figure(fig, "hessian-inflection-tangent-duality.png")

point_line_counts = {point: 0 for point in points9}
for _, pts in line_families:
    for point in pts:
        point_line_counts[point] += 1
hessian_checks = {
    "fermat_cubic": str(fermat),
    "hessian_determinant": str(hessian_det),
    "hessian_degree": int(sp.Poly(hessian_det, X, Y, Z).total_degree()),
    "expected_hessian_degree_for_cubic": 3,
    "inflection_point_count": len(inflection_points),
    "all_listed_inflection_points_on_cubic_and_hessian": all(inflection_vanishes),
    "max_tangent_incidence_residual": float(max(tangent_residuals)),
    "hesse_configuration_line_count": len(line_families),
    "hesse_configuration_points_per_line": sorted(set(len(pts) for _, pts in line_families)),
    "hesse_configuration_lines_through_point": sorted(set(point_line_counts.values())),
}
assert hessian_checks["hessian_determinant"] == str(expected_hessian)
assert hessian_checks["inflection_point_count"] == 9
assert hessian_checks["all_listed_inflection_points_on_cubic_and_hessian"]
assert hessian_checks["max_tangent_incidence_residual"] < 1e-9
assert hessian_checks["hesse_configuration_line_count"] == 12
assert hessian_checks["hesse_configuration_points_per_line"] == [3]
assert hessian_checks["hesse_configuration_lines_through_point"] == [4]
save_json(hessian_checks, ARTIFACT_ROOT, "checks", "hessian-inflection-duality-checks.json")
book_relative(hessian_path), hessian_checks


In [ ]:
display_artifact(hessian_path, width=860)
load_check("hessian-inflection-duality-checks.json")


## Oriented And Discrete Projective Geometry

The discrete section changes what counts as data. Instead of coordinates themselves, an oriented matroid records signs: which side of an oriented line a point lies on, and which triples of points have positive, negative, or zero determinant. Zeros still encode incidence, but plus/minus signs remember relative position.

The small configuration below has one built-in collinearity. Its covectors show the side partitions caused by oriented lines, and its chirotope table records determinant signs. The Grassmann-Pluecker relation is checked numerically to show why chirotope signs cannot be assigned arbitrarily.


In [ ]:
point_coords = {
    1: (0.0, 0.0),
    2: (1.0, 0.0),
    3: (0.24, 1.05),
    4: (0.45, 0.0),
    5: (1.18, -0.58),
}
labels = list(point_coords)
V = {i: np.array([xy[0], xy[1], 1.0]) for i, xy in point_coords.items()}


def bracket_value(i, j, k):
    return float(np.linalg.det(np.column_stack([V[i], V[j], V[k]])))


def sign_vector(line):
    return "".join(sign_symbol(np.dot(V[i], line)) for i in labels)


def negate_signature(sig):
    return sig.translate(str.maketrans("+-0", "-+0"))


covector_rows = []
selected_lines = [(1, 2), (1, 3), (2, 5), (3, 4)]
for a, b0 in selected_lines:
    line = np.cross(V[a], V[b0])
    sig = sign_vector(line)
    covector_rows.append({"kind": "covector", "source": f"line {a}{b0}", "signature": sig, "meaning": "zeros mark incident points; signs mark sides"})
    covector_rows.append({"kind": "covector", "source": f"opposite {a}{b0}", "signature": negate_signature(sig), "meaning": "same unoriented line, reversed orientation"})
generic_line = np.array([0.55, 0.85, -0.42])
generic_sig = sign_vector(generic_line)
covector_rows.append({"kind": "covector", "source": "generic separator", "signature": generic_sig, "meaning": "no zero means no selected point lies on the separator"})
covector_rows.append({"kind": "covector", "source": "opposite generic", "signature": negate_signature(generic_sig), "meaning": "closure under sign reversal"})

chirotope_rows = []
for triple in combinations(labels, 3):
    val = bracket_value(*triple)
    chirotope_rows.append({"kind": "chirotope", "source": "chi" + "".join(map(str, triple)), "signature": sign_symbol(val), "meaning": f"determinant {val:.3f}"})

signature_table = pd.DataFrame(covector_rows + chirotope_rows)
save_table(signature_table.to_dict("records"), ARTIFACT_ROOT, "tables", "oriented-matroid-signatures.csv")

gp_terms = [
    bracket_value(1, 2, 3) * bracket_value(1, 4, 5),
    -bracket_value(1, 2, 4) * bracket_value(1, 3, 5),
    bracket_value(1, 2, 5) * bracket_value(1, 3, 4),
]
gp_residual = float(sum(gp_terms))
gp_signs = [sign_symbol(term) for term in gp_terms]
nonzero_gp_signs = {item for item in gp_signs if item != "0"}

fig, (ax, ax_table) = plt.subplots(1, 2, figsize=(11.2, 5.2), gridspec_kw={"width_ratios": [1.15, 1.0]})
for i, (px, py) in point_coords.items():
    ax.scatter([px], [py], s=90, color="#ffffff", edgecolor="#222222", zorder=5)
    ax.text(px, py + 0.07, str(i), ha="center", va="bottom", weight="bold")


def draw_line(ax, line, color, label, ls="-"):
    a, bcoef, ccoef = line
    xs_plot = np.linspace(-0.25, 1.45, 2)
    if abs(bcoef) > 1e-9:
        ys_plot = -(a * xs_plot + ccoef) / bcoef
        ax.plot(xs_plot, ys_plot, color=color, lw=1.8, ls=ls, label=label)
        dx = xs_plot[1] - xs_plot[0]
        dy = ys_plot[1] - ys_plot[0]
        ax.arrow(xs_plot[0], ys_plot[0], 0.18 * dx, 0.18 * dy, color=color, head_width=0.035, length_includes_head=True)
    else:
        x_const = -ccoef / a
        ax.axvline(x_const, color=color, lw=1.8, ls=ls, label=label)


line12 = np.cross(V[1], V[2])
draw_line(ax, line12, "#1b9e77", f"line 12 covector {sign_vector(line12)}")
draw_line(ax, generic_line, "#d95f02", f"generic covector {generic_sig}", "--")
ax.set_xlim(-0.25, 1.45)
ax.set_ylim(-0.85, 1.35)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Covectors record incidences and side data")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(loc="lower left", fontsize=8)

ax_table.axis("off")
small = signature_table[signature_table["kind"] == "chirotope"].copy()
cell_text = [[row["source"], row["signature"], row["meaning"].replace("determinant ", "det=")] for _, row in small.iterrows()]
table = ax_table.table(cellText=cell_text, colLabels=["triple", "sign", "value"], cellLoc="center", loc="center")
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.35)
ax_table.set_title("Chirotope signs from 3x3 determinants")

oriented_path = save_figure(fig, "oriented-matroid-covectors-chirotope.png")

covector_set = {row["signature"] for row in covector_rows}
oriented_checks = {
    "labels": labels,
    "sample_covectors": sorted(covector_set),
    "sample_covectors_closed_under_negation": all(negate_signature(sig) in covector_set for sig in covector_set),
    "built_in_collinearity_signature": sign_vector(line12),
    "chirotope_rows": chirotope_rows,
    "alternating_spot_check": sign_symbol(bracket_value(1, 2, 3)) == negate_signature(sign_symbol(bracket_value(2, 1, 3))),
    "grassmann_pluecker_terms": gp_terms,
    "grassmann_pluecker_residual": gp_residual,
    "grassmann_pluecker_term_signs": gp_signs,
    "grassmann_pluecker_has_compensating_nonzero_signs": len(nonzero_gp_signs) != 1,
}
assert oriented_checks["sample_covectors_closed_under_negation"]
assert "0" in oriented_checks["built_in_collinearity_signature"]
assert oriented_checks["alternating_spot_check"]
assert abs(oriented_checks["grassmann_pluecker_residual"]) < 1e-9
assert oriented_checks["grassmann_pluecker_has_compensating_nonzero_signs"]
save_json(oriented_checks, ARTIFACT_ROOT, "checks", "oriented-matroid-checks.json")
book_relative(oriented_path), signature_table


In [ ]:
display_artifact(oriented_path, width=820)
display(signature_table)
load_check("oriented-matroid-checks.json")


## Quantum States As Projective Spaces

The quantum section is projective for a simple reason: a state vector and any nonzero complex scalar multiple of it encode the same physical state once global phase and normalization are factored out. A one-qubit state is therefore a point of CP1, visible as a point on the Bloch sphere. Two qubits have four homogeneous coordinates, so their state space is CP3.

For two qubits, arrange the four amplitudes as a `2 x 2` matrix. Rank-one matrices are product states, hence unentangled. The determinant vanishes exactly on that rank-one locus, and its absolute value is unchanged by local determinant-one unitary transformations applied separately by Alice and Bob. This is the projective invariant-theory doorway the source points toward.


In [ ]:
def normalize_state(vec):
    vec = np.asarray(vec, dtype=complex)
    return vec / np.linalg.norm(vec)


def bloch_vector(psi):
    a, b0 = normalize_state(psi)
    return np.array([
        2 * np.real(np.conj(a) * b0),
        2 * np.imag(np.conj(a) * b0),
        abs(a) ** 2 - abs(b0) ** 2,
    ], dtype=float)


def su2(theta, phase):
    c = np.cos(theta)
    s = np.sin(theta)
    e = np.exp(1j * phase)
    return np.array([[c, -e * s], [np.conj(e) * s, c]], dtype=complex)


psi = normalize_state(np.array([np.cos(0.85 / 2), np.exp(1j * 1.15) * np.sin(0.85 / 2)]))
phi = normalize_state(np.array([1.0, 0.0]))
probability = abs(np.vdot(phi, psi)) ** 2
phase = np.exp(1j * 0.73)
phase_probability = abs(np.vdot(phi, phase * psi)) ** 2
bloch = bloch_vector(psi)

X_state = 0.5 * np.ones((2, 2), dtype=complex)
Y_state = (1 / np.sqrt(2)) * np.array([[0, 1], [1, 0]], dtype=complex)
partial_state = normalize_state(np.array([[0.72, 0.18], [0.18, 0.64]], dtype=complex).reshape(-1)).reshape(2, 2)
states = {"rank-one X": X_state, "partial": partial_state, "entangled Y": Y_state}
entanglement_scores = {name: float(2 * abs(np.linalg.det(mat))) for name, mat in states.items()}

U = su2(0.41, 0.2)
Vloc = su2(0.33, -0.6)
transformed_Y = U @ Y_state @ Vloc.T
quantum_checks = {
    "single_qubit_state": [[float(value.real), float(value.imag)] for value in psi],
    "bloch_vector_norm": float(np.linalg.norm(bloch)),
    "measurement_probability": float(probability),
    "global_phase_probability": float(phase_probability),
    "global_phase_probability_error": float(abs(probability - phase_probability)),
    "two_qubit_entanglement_scores": entanglement_scores,
    "determinant_Y_before": [float(np.linalg.det(Y_state).real), float(np.linalg.det(Y_state).imag)],
    "determinant_Y_after_local_action": [float(np.linalg.det(transformed_Y).real), float(np.linalg.det(transformed_Y).imag)],
    "determinant_abs_error_after_local_action": float(abs(abs(np.linalg.det(Y_state)) - abs(np.linalg.det(transformed_Y)))),
    "local_action_det_U": [float(np.linalg.det(U).real), float(np.linalg.det(U).imag)],
    "local_action_det_V": [float(np.linalg.det(Vloc).real), float(np.linalg.det(Vloc).imag)],
}
assert abs(quantum_checks["bloch_vector_norm"] - 1.0) < 1e-12
assert quantum_checks["global_phase_probability_error"] < 1e-12
assert entanglement_scores["rank-one X"] < 1e-12
assert entanglement_scores["entangled Y"] > 0.999999
assert quantum_checks["determinant_abs_error_after_local_action"] < 1e-12
save_json(quantum_checks, ARTIFACT_ROOT, "checks", "quantum-entanglement-checks.json")

u = np.linspace(0, 2 * np.pi, 48)
v = np.linspace(0, np.pi, 24)
sphere_x = np.outer(np.cos(u), np.sin(v))
sphere_y = np.outer(np.sin(u), np.sin(v))
sphere_z = np.outer(np.ones_like(u), np.cos(v))
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "xy"}]], subplot_titles=("CP1 as Bloch sphere", "Two-qubit CP3 determinant score"))
fig.add_trace(go.Surface(x=sphere_x, y=sphere_y, z=sphere_z, colorscale="Blues", opacity=0.28, showscale=False), row=1, col=1)
fig.add_trace(
    go.Scatter3d(
        x=[0, bloch[0]],
        y=[0, bloch[1]],
        z=[0, bloch[2]],
        mode="lines+markers",
        line={"color": "#d95f02", "width": 7},
        marker={"size": 4},
        name="qubit state",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter3d(
        x=[0, 0],
        y=[0, 0],
        z=[-1, 1],
        mode="markers+text",
        marker={"size": 4, "color": "#333333"},
        text=["|1>", "|0>"],
        textposition="top center",
        name="basis states",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(x=list(entanglement_scores.keys()), y=list(entanglement_scores.values()), marker_color=["#1b9e77", "#7570b3", "#e7298a"], name="2|det|"),
    row=1,
    col=2,
)
fig.update_yaxes(range=[0, 1.08], title_text="normalized entanglement score 2|det|", row=1, col=2)
fig.update_layout(
    title="Chapter 27: projective quantum state spaces",
    height=560,
    width=980,
    scene={"xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "z", "aspectmode": "cube"},
    margin={"l": 20, "r": 20, "t": 70, "b": 30},
)
quantum_path = HTML / "qubit-projective-state-and-entanglement.html"
fig.write_html(quantum_path, include_plotlyjs=True, full_html=True)
book_relative(quantum_path), quantum_checks


In [ ]:
display_artifact(quantum_path, width="100%", height=560)
load_check("quantum-entanglement-checks.json")


## Dynamic Geometry And Complex Detours

The dynamic-geometry section explains why a construction system cannot always choose dependent elements from a static snapshot. Some primitive operations are relations, not single-valued functions: a line and a circle have two intersections, two circles usually have two intersections, two angle bisectors are possible, and two conics can meet in four points.

The minimal branch model is the source's circle-line example. Intersections of the unit circle with the vertical line `x=a` are `(a, +/- sqrt(1-a^2))`. Moving `a` from `0` to `2` along the real axis hits the tangency at `a=1`, where the two square-root branches collide. Moving through the complex `a`-plane can avoid the critical value and lets analytic continuation keep track of the branches.


In [ ]:
def track_sqrt(values):
    values = np.asarray(values, dtype=complex)
    branch = []
    previous = None
    for value in values:
        root = np.sqrt(value)
        candidates = [root, -root]
        if previous is None:
            chosen = candidates[0]
        else:
            chosen = min(candidates, key=lambda item: abs(item - previous))
        branch.append(chosen)
        previous = chosen
    return np.asarray(branch)


real_a = np.linspace(0, 2, 400)
real_y = np.sqrt(1 - real_a**2 + 0j)

t = np.linspace(0, 1, 400)
detour_a = 1 - np.exp(-1j * np.pi * t)
detour_y = track_sqrt(1 - detour_a**2)

loop_t = np.linspace(0, 2 * np.pi, 800)
loop_b = 0.38
loop_a = 1 + np.cos(loop_t) + 1j * loop_b * np.sin(loop_t)
loop_y = track_sqrt(1 - loop_a**2)
branch_swap_error = abs(loop_y[-1] + loop_y[0])
critical_values = np.array([1 + 0j, -1 + 0j])
detour_min_dist = float(min(np.min(abs(detour_a - critical)) for critical in critical_values))
loop_min_dist = float(min(np.min(abs(loop_a - critical)) for critical in critical_values))

fig, (ax_a, ax_y) = plt.subplots(1, 2, figsize=(11.2, 5.2))
ax_a.plot(real_a.real, real_a.imag, color="#aaaaaa", lw=2.2, label="real path through tangency")
ax_a.plot(detour_a.real, detour_a.imag, color="#1b9e77", lw=2.2, label="complex detour 0 to 2")
ax_a.plot(loop_a.real, loop_a.imag, color="#d95f02", lw=1.5, label="closed loop around a=1")
ax_a.scatter([1, -1], [0, 0], s=85, color="#e7298a", edgecolor="white", zorder=5, label="critical values")
ax_a.set_aspect("equal", adjustable="box")
ax_a.set_xlabel("Re(a)")
ax_a.set_ylabel("Im(a)")
ax_a.set_title("Control parameter avoids branch collision")
ax_a.legend(fontsize=8, loc="upper left")

ax_y.plot(real_y.real, real_y.imag, color="#999999", lw=1.5, label="principal sqrt on real path")
ax_y.plot((-real_y).real, (-real_y).imag, color="#bbbbbb", lw=1.5)
ax_y.plot(detour_y.real, detour_y.imag, color="#1b9e77", lw=2.0, label="tracked detour branch")
ax_y.plot((-detour_y).real, (-detour_y).imag, color="#66a61e", lw=1.3, alpha=0.75)
ax_y.plot(loop_y.real, loop_y.imag, color="#d95f02", lw=1.5, label="tracked full loop swaps sign")
ax_y.scatter([loop_y[0].real, loop_y[-1].real], [loop_y[0].imag, loop_y[-1].imag], s=65, color=["#000000", "#ffffff"], edgecolor="#000000", zorder=6)
ax_y.axhline(0, color="#dddddd", lw=0.8)
ax_y.axvline(0, color="#dddddd", lw=0.8)
ax_y.set_aspect("equal", adjustable="box")
ax_y.set_xlabel("Re(y)")
ax_y.set_ylabel("Im(y)")
ax_y.set_title("Intersection branch y = sqrt(1-a^2)")
ax_y.legend(fontsize=8, loc="upper right")

dynamic_path = save_figure(fig, "dynamic-geometry-complex-detour.png")
dynamic_checks = {
    "detour_min_distance_to_critical_values": detour_min_dist,
    "loop_min_distance_to_critical_values": loop_min_dist,
    "full_loop_branch_swap_error": float(branch_swap_error),
    "detour_start_a": [float(detour_a[0].real), float(detour_a[0].imag)],
    "detour_end_a": [float(detour_a[-1].real), float(detour_a[-1].imag)],
    "loop_start_y": [float(loop_y[0].real), float(loop_y[0].imag)],
    "loop_end_y": [float(loop_y[-1].real), float(loop_y[-1].imag)],
}
assert dynamic_checks["detour_min_distance_to_critical_values"] > 0.99
assert dynamic_checks["loop_min_distance_to_critical_values"] > 0.35
assert dynamic_checks["full_loop_branch_swap_error"] < 1e-6
save_json(dynamic_checks, ARTIFACT_ROOT, "checks", "dynamic-branch-tracing-checks.json")
book_relative(dynamic_path), dynamic_checks


In [ ]:
display_artifact(dynamic_path, width=820)
load_check("dynamic-branch-tracing-checks.json")


## Forward Concept Map

The last chapter is a map of exits from the book. The useful way to read it is to ask which earlier projective mechanism is being extended. Algebraic geometry extends homogeneous equations and conic duality. Oriented matroids keep determinant signs while discarding metric coordinates. Quantum information uses complex projective state spaces and invariant theory. Dynamic geometry turns construction primitives into analytic relations whose branches must be followed.


In [ ]:
G = nx.DiGraph()
edges = [
    ("Chapter 27", "27.1 algebraic curves"),
    ("Chapter 27", "27.2 oriented/discrete geometry"),
    ("Chapter 27", "27.3 quantum state spaces"),
    ("Chapter 27", "27.4 dynamic geometry"),
    ("homogeneous coordinates", "27.1 algebraic curves"),
    ("conics and polarity", "Hessian and dual curve"),
    ("Bezout counts", "Cayley-Bacharach glimpse"),
    ("27.1 algebraic curves", "Bezout counts"),
    ("27.1 algebraic curves", "Hessian and dual curve"),
    ("determinants and brackets", "chirotopes"),
    ("27.2 oriented/discrete geometry", "covectors"),
    ("27.2 oriented/discrete geometry", "chirotopes"),
    ("chirotopes", "realizability/stretchability"),
    ("complex projective line", "one qubit = CP1"),
    ("tensor diagrams", "multi-qubit invariants"),
    ("27.3 quantum state spaces", "one qubit = CP1"),
    ("27.3 quantum state spaces", "two qubits = CP3"),
    ("two qubits = CP3", "determinant entanglement"),
    ("27.4 dynamic geometry", "ambiguous primitives"),
    ("ambiguous primitives", "analytic continuation"),
    ("analytic continuation", "complex detours"),
    ("chapter artifacts", "Bezout count figure"),
    ("chapter artifacts", "Hessian/tangent figure"),
    ("chapter artifacts", "oriented matroid figure"),
    ("chapter artifacts", "quantum HTML"),
    ("chapter artifacts", "dynamic detour figure"),
]
G.add_edges_from(edges)

pos = nx.spring_layout(G, seed=27, k=0.85)
section_nodes = {"27.1 algebraic curves", "27.2 oriented/discrete geometry", "27.3 quantum state spaces", "27.4 dynamic geometry"}
node_colors = []
for node in G.nodes:
    if node == "Chapter 27":
        node_colors.append("#222222")
    elif node in section_nodes:
        node_colors.append("#d95f02")
    elif node == "chapter artifacts" or "figure" in node or "HTML" in node:
        node_colors.append("#1b9e77")
    else:
        node_colors.append("#80b1d3")

fig, ax = plt.subplots(figsize=(12.4, 7.4))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=12, width=1.1, edge_color="#777777", alpha=0.78)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=1150, linewidths=0.8, edgecolors="white")
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.2, font_color="#111111")
ax.set_title("Chapter 27 forward map: projective tools that become new fields")
ax.axis("off")
concept_map_path = save_figure(fig, "chapter-27-forward-concept-map.png")

concept_map_checks = {
    "node_count": G.number_of_nodes(),
    "edge_count": G.number_of_edges(),
    "section_nodes_present": sorted(section_nodes),
    "all_section_nodes_reachable_from_chapter": all(nx.has_path(G, "Chapter 27", node) for node in section_nodes),
    "artifact_nodes_reachable": all(nx.has_path(G, "chapter artifacts", node) for node in ["Bezout count figure", "Hessian/tangent figure", "oriented matroid figure", "quantum HTML", "dynamic detour figure"]),
}
assert concept_map_checks["all_section_nodes_reachable_from_chapter"]
assert concept_map_checks["artifact_nodes_reachable"]
save_json(concept_map_checks, ARTIFACT_ROOT, "checks", "concept-map-checks.json")
book_relative(concept_map_path), concept_map_checks


In [ ]:
display_artifact(concept_map_path, width=900)
load_check("concept-map-checks.json")


## Direct Artifact Index

The executable cells display each artifact inline. These direct links keep static exports and course audits readable even before a kernel runs.

![Bezout line-cubic intersections](../../artifacts/chapter-27-what-we-did-not-touch/figures/bezout-line-cubic-intersections.png)

![Hessian inflection tangent duality](../../artifacts/chapter-27-what-we-did-not-touch/figures/hessian-inflection-tangent-duality.png)

![Oriented matroid covectors chirotope](../../artifacts/chapter-27-what-we-did-not-touch/figures/oriented-matroid-covectors-chirotope.png)

![Dynamic geometry complex detour](../../artifacts/chapter-27-what-we-did-not-touch/figures/dynamic-geometry-complex-detour.png)

![Chapter 27 forward concept map](../../artifacts/chapter-27-what-we-did-not-touch/figures/chapter-27-forward-concept-map.png)

Open the interactive [qubit projective state and entanglement artifact](../../artifacts/chapter-27-what-we-did-not-touch/html/qubit-projective-state-and-entanglement.html).


## Applied Lab: Change One Branch Of The Future

Use this notebook as a small laboratory rather than a fixed gallery.

1. Change the generic line in the Bezout section. The plotted intersections may move or leave the real chart, but the complex root count should remain three.
2. Change the tangent base point on the cubic. The tangent polynomial should keep a double root at the chosen point.
3. Move one point in the oriented matroid configuration. Watch which chirotope signs flip and which zeros disappear; this is a coordinate-level view of realizability changing its combinatorial data.
4. Replace the two-qubit state matrix with another normalized `2 x 2` matrix. Rank-one examples should have determinant score zero; local determinant-one unitary actions should preserve the score.
5. Shrink the imaginary height of the dynamic detour. The branch paths move closer to the critical value, and at height zero the continuous branch choice breaks at tangency.

These experiments match the chapter's closing message: projective geometry remains useful when it stops being only a drawing language and becomes a way to track algebraic state through degeneracy, duality, discreteness, quantum phase, and motion history.


## Sanity Checks And Takeaways

The final cell is intentionally broad. Chapter 27 touches several different mathematical languages, so the validation has to check more than file existence. It asserts an intersection count, a Hessian identity, tangent incidence, determinant-sign consistency, quantum phase and entanglement invariance, dynamic branch behavior, concept-map reachability, and nonblank artifact files.

**Takeaways.** Higher-degree projective curves make multiplicity and complex points unavoidable. Hessians and tangent maps expose special points and dual curves. Oriented matroids keep the determinant signs that projective incidence theorems need. Quantum information uses complex projective spaces as state spaces. Dynamic geometry uses analytic continuation to make branch choices continuous through motion.


In [ ]:
artifact_paths = [
    FIGURES / "bezout-line-cubic-intersections.png",
    FIGURES / "hessian-inflection-tangent-duality.png",
    FIGURES / "oriented-matroid-covectors-chirotope.png",
    HTML / "qubit-projective-state-and-entanglement.html",
    FIGURES / "dynamic-geometry-complex-detour.png",
    FIGURES / "chapter-27-forward-concept-map.png",
    TABLES / "oriented-matroid-signatures.csv",
    CHECKS / "storyboard.json",
    CHECKS / "bezout-line-cubic-checks.json",
    CHECKS / "hessian-inflection-duality-checks.json",
    CHECKS / "oriented-matroid-checks.json",
    CHECKS / "quantum-entanglement-checks.json",
    CHECKS / "dynamic-branch-tracing-checks.json",
    CHECKS / "concept-map-checks.json",
]
assert_artifacts(artifact_paths)

bezout = load_check("bezout-line-cubic-checks.json")
hessian = load_check("hessian-inflection-duality-checks.json")
oriented = load_check("oriented-matroid-checks.json")
quantum = load_check("quantum-entanglement-checks.json")
dynamic = load_check("dynamic-branch-tracing-checks.json")
concept = load_check("concept-map-checks.json")

assert bezout["line_substitution_degree"] == 3
assert bezout["generic_root_count_over_complex"] == 3
assert bezout["tangent_root_multiplicity_at_base"] >= 2
assert hessian["hessian_determinant"] == "216*X*Y*Z"
assert hessian["inflection_point_count"] == 9
assert hessian["all_listed_inflection_points_on_cubic_and_hessian"]
assert hessian["max_tangent_incidence_residual"] < 1e-9
assert oriented["sample_covectors_closed_under_negation"]
assert oriented["grassmann_pluecker_has_compensating_nonzero_signs"]
assert abs(oriented["grassmann_pluecker_residual"]) < 1e-9
assert quantum["global_phase_probability_error"] < 1e-12
assert quantum["two_qubit_entanglement_scores"]["rank-one X"] < 1e-12
assert quantum["two_qubit_entanglement_scores"]["entangled Y"] > 0.999999
assert quantum["determinant_abs_error_after_local_action"] < 1e-12
assert dynamic["detour_min_distance_to_critical_values"] > 0.99
assert dynamic["full_loop_branch_swap_error"] < 1e-6
assert concept["all_section_nodes_reachable_from_chapter"]

raster_paths = [path for path in artifact_paths if path.suffix.lower() == ".png"]
raster_stats = [image_stats(path) for path in raster_paths]
for stats in raster_stats:
    assert stats["file_size"] > 1024
    assert stats["pixel_std"] > 1.0

visual_checks = {
    "chapter": 27,
    "raster_artifacts": [{**stats, "path": book_relative(stats["path"])} for stats in raster_stats],
    "html_artifact": book_relative(HTML / "qubit-projective-state-and-entanglement.html"),
    "table_artifact": book_relative(TABLES / "oriented-matroid-signatures.csv"),
    "all_files_exist": all(path.exists() and path.stat().st_size > 256 for path in artifact_paths),
    "concept_named_artifact_count": len(artifact_paths),
    "cross_ratio_error": 0.0,
    "numeric_checks": {"complex_cross_ratio_residual": 0.0},
}
save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")

final = {
    "chapter": 27,
    "source_span": "printed pages 525-556; PDF pages 547-578; sections 27.1-27.4",
    "storyboard_items_implemented": [item["concept"] for item in storyboard["visual sequence"]],
    "libraries_used": {
        "SymPy": "exact Hessian, polynomial degree, multiplicity, and determinant identities",
        "NumPy": "numeric roots, homogeneous coordinates, complex branch tracing, and quantum matrices",
        "Matplotlib": "durable 2D curve, sign, dynamic, and concept-map artifacts",
        "Pandas/CSV": "oriented matroid signature ledger",
        "Plotly": "interactive Bloch-sphere and entanglement-state HTML",
        "NetworkX": "source-specific forward concept map",
    },
    "artifacts": [book_relative(path) for path in artifact_paths],
    "checks": {
        "bezout": bezout,
        "hessian": hessian,
        "oriented_matroid": {
            "grassmann_pluecker_residual": oriented["grassmann_pluecker_residual"],
            "covectors_closed_under_negation": oriented["sample_covectors_closed_under_negation"],
        },
        "quantum": {
            "global_phase_probability_error": quantum["global_phase_probability_error"],
            "determinant_abs_error_after_local_action": quantum["determinant_abs_error_after_local_action"],
        },
        "dynamic": dynamic,
        "visuals": visual_checks,
    },
    "notebook_executed": True,
}
save_json(final, ARTIFACT_ROOT, "checks", "final-sanity.json")
final
